# Ensemble Methods — Gradient Boosting

Gradient boosting (Friedman, 2001) builds an additive model $F_M(x) = F_0(x) + \nu\sum_{m=1}^{M} h_m(x)$
by **gradient descent in function space**:

1. $F_0 = \arg\min_c \sum_i L(y_i, c)$ (a constant).
2. For $m = 1..M$:
   * Compute **pseudo-residuals** (negative gradient): $r_{i} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$
   * Fit a small regression tree $h_m$ to $\{(x_i, r_i)\}$.
   * For each leaf $j$, choose the best output $\gamma_{j}$ for the actual loss.
   * Update $F_m = F_{m-1} + \nu \cdot h_m$ with learning rate (shrinkage) $\nu$.

| Loss | $L(y,F)$ | Pseudo-residual | Leaf value $\gamma_j$ |
|---|---|---|---|
| Squared error (regression) | $\frac12(y-F)^2$ | $y - F$ | mean of residuals in leaf |
| Log-loss (binary classif.) | $-[y\log p + (1-y)\log(1-p)]$, $p=\sigma(F)$ | $y - p$ | $\frac{\sum r_i}{\sum p_i(1-p_i)}$ (Newton step) |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Base learner: compact CART regression tree

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value                     # class-probability vector or regression value
        self.feature = self.threshold = self.left = self.right = None
        self.leaf_id = None


class DecisionTree:
    """Compact CART (see 06_Decision_Tree_CART) used as the base learner.

    Classification labels must be integers 0..K-1; leaves store class probabilities.
    max_features: None (all), int, or "sqrt" -> random feature subset at every split (Random Forest).
    """

    def __init__(self, task="classification", max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, max_features=None, random_state=None):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.rng = np.random.RandomState(random_state)

    def fit(self, X, y, n_classes=None):
        X = np.asarray(X, dtype=float)
        if self.task == "classification":
            y = np.asarray(y, dtype=int)
            self.n_classes = n_classes or y.max() + 1
        else:
            y = np.asarray(y, dtype=float)
        d = X.shape[1]
        if self.max_features == "sqrt":
            self.k_features = max(1, int(np.sqrt(d)))
        else:
            self.k_features = d if self.max_features is None else self.max_features
        self.feature_importances_ = np.zeros(d)
        self.leaves_ = []
        self.root = self._build(X, y, 0)
        if self.feature_importances_.sum() > 0:
            self.feature_importances_ /= self.feature_importances_.sum()
        return self

    def _impurity(self, y):
        if self.task == "classification":
            p = np.bincount(y, minlength=self.n_classes) / len(y)
            return 1 - np.sum(p ** 2)
        return y.var()

    def _leaf(self, node):
        node.leaf_id = len(self.leaves_)
        self.leaves_.append(node)
        return node

    def _best_split(self, X, y):
        n, d = X.shape
        features = self.rng.choice(d, self.k_features, replace=False)
        nl = np.arange(1, n)
        nr = n - nl
        best = (None, None, np.inf)
        for f in features:
            order = np.argsort(X[:, f], kind="mergesort")
            xs, ys = X[order, f], y[order]
            if self.task == "classification":
                onehot = np.eye(self.n_classes)[ys]
                left = np.cumsum(onehot, axis=0)[:-1]
                right = onehot.sum(axis=0) - left
                cost = (nl * (1 - np.sum((left / nl[:, None]) ** 2, 1))
                        + nr * (1 - np.sum((right / nr[:, None]) ** 2, 1))) / n
            else:
                cs, cs2 = np.cumsum(ys)[:-1], np.cumsum(ys ** 2)[:-1]
                cost = ((cs2 - cs ** 2 / nl) + ((ys ** 2).sum() - cs2) - (ys.sum() - cs) ** 2 / nr) / n
            valid = (xs[1:] != xs[:-1]) & (nl >= self.min_samples_leaf) & (nr >= self.min_samples_leaf)
            if valid.any():
                cost = np.where(valid, cost, np.inf)
                i = np.argmin(cost)
                if cost[i] < best[2]:
                    best = (f, (xs[i] + xs[i + 1]) / 2, cost[i])
        return best

    def _build(self, X, y, depth):
        if self.task == "classification":
            node = Node(np.bincount(y, minlength=self.n_classes) / len(y))
        else:
            node = Node(y.mean())
        impurity = self._impurity(y)
        if (impurity < 1e-12 or len(y) < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            return self._leaf(node)
        f, t, cost = self._best_split(X, y)
        if f is None:
            return self._leaf(node)
        self.feature_importances_[f] += len(y) * (impurity - cost)    # weighted impurity decrease
        mask = X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

    def _route(self, node, X, idx, out):
        if node.left is None:
            out[idx] = node.leaf_id
            return
        mask = X[idx, node.feature] <= node.threshold
        self._route(node.left, X, idx[mask], out)
        self._route(node.right, X, idx[~mask], out)

    def apply(self, X):
        """Index (into self.leaves_) of the leaf each sample lands in."""
        X = np.asarray(X, dtype=float)
        out = np.empty(len(X), dtype=int)
        self._route(self.root, X, np.arange(len(X)), out)
        return out

    def predict_proba(self, X):
        return np.array([leaf.value for leaf in self.leaves_])[self.apply(X)]

    def predict(self, X):
        values = self.predict_proba(X)
        return values.argmax(axis=1) if self.task == "classification" else values

## Gradient boosting for regression (squared loss)

In [ ]:
class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample           # < 1.0 gives *stochastic* gradient boosting
        self.rng = np.random.RandomState(random_state)

    def fit(self, X, y):
        n = len(y)
        self.F0 = y.mean()
        F = np.full(n, self.F0)
        self.trees, self.train_loss = [], []
        for _ in range(self.n_estimators):
            residuals = y - F                                  # negative gradient of 1/2 (y - F)^2
            idx = self.rng.choice(n, int(self.subsample * n), replace=False)
            tree = DecisionTree(task="regression", max_depth=self.max_depth).fit(X[idx], residuals[idx])
            F += self.learning_rate * tree.predict(X)
            self.trees.append(tree)
            self.train_loss.append(np.mean((y - F) ** 2))
        return self

    def staged_predict(self, X):
        F = np.full(len(X), self.F0)
        for tree in self.trees:
            F = F + self.learning_rate * tree.predict(X)
            yield F

    def predict(self, X):
        *_, F = self.staged_predict(X)
        return F

In [ ]:
rng = np.random.RandomState(0)
x = np.sort(rng.uniform(0, 6, 300))
y = np.sin(x) + 0.3 * np.cos(3 * x) + rng.normal(0, 0.2, len(x))
X = x.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
grid = np.linspace(0, 6, 500).reshape(-1, 1)

gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=2).fit(X_train, y_train)
stages = list(gbr.staged_predict(grid))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.3), sharey=True)
for ax, m in zip(axes, [1, 5, 25, 200]):
    ax.scatter(X_train, y_train, s=8, c="gray")
    ax.plot(grid, stages[m - 1], "r", lw=2)
    ax.set_title(f"after {m} tree(s)")
plt.tight_layout(); plt.show()

### What does each tree learn?
The first trees fit the residuals left by $F_0$; each later tree fits what is still unexplained.

In [ ]:
F_prev = np.full(len(X_train), gbr.F0)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for m, ax in enumerate(axes):
    residuals = y_train - F_prev
    order = np.argsort(X_train[:, 0])
    ax.scatter(X_train[:, 0], residuals, s=8, c="gray", label="residuals")
    ax.plot(X_train[order, 0], gbr.trees[m].predict(X_train[order]), "g", lw=2, label=f"tree {m + 1}")
    ax.set_title(f"Tree {m + 1} fits the residuals of F_{m}"); ax.legend()
    F_prev = F_prev + gbr.learning_rate * gbr.trees[m].predict(X_train)
plt.tight_layout(); plt.show()

### Learning rate (shrinkage) vs number of trees

In [ ]:
plt.figure(figsize=(8, 5))
for lr in [1.0, 0.3, 0.1, 0.03]:
    m = GradientBoostingRegressor(n_estimators=300, learning_rate=lr, max_depth=2, random_state=0).fit(X_train, y_train)
    test_mse = [np.mean((y_test - F) ** 2) for F in m.staged_predict(X_test)]
    plt.plot(test_mse, label=f"lr={lr} (best {min(test_mse):.4f} @ {np.argmin(test_mse) + 1})")
plt.axhline(0.04, ls=":", c="gray", label="noise variance")
plt.yscale("log"); plt.xlabel("number of trees"); plt.ylabel("test MSE"); plt.legend()
plt.title("Smaller learning rates need more trees but overfit less"); plt.show()

## Gradient boosting for binary classification (log-loss)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


class GradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample
        self.rng = np.random.RandomState(random_state)

    def fit(self, X, y):
        n = len(y)
        p0 = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.F0 = np.log(p0 / (1 - p0))                # log-odds of the base rate
        F = np.full(n, self.F0)
        self.trees, self.train_loss = [], []
        for _ in range(self.n_estimators):
            p = sigmoid(F)
            residuals = y - p                            # negative gradient of the log-loss
            idx = self.rng.choice(n, int(self.subsample * n), replace=False)
            tree = DecisionTree(task="regression", max_depth=self.max_depth).fit(X[idx], residuals[idx])

            # replace each leaf's value by a Newton step on the log-loss
            leaf_of = tree.apply(X[idx])
            for j, leaf in enumerate(tree.leaves_):
                in_leaf = idx[leaf_of == j]
                num = residuals[in_leaf].sum()
                den = (p[in_leaf] * (1 - p[in_leaf])).sum()
                leaf.value = num / den if den > 1e-12 else 0.0

            F += self.learning_rate * tree.predict(X)
            self.trees.append(tree)
            p = np.clip(sigmoid(F), 1e-12, 1 - 1e-12)
            self.train_loss.append(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
        return self

    def staged_decision_function(self, X):
        F = np.full(len(X), self.F0)
        for tree in self.trees:
            F = F + self.learning_rate * tree.predict(X)
            yield F

    def predict_proba(self, X):
        *_, F = self.staged_decision_function(X)
        return sigmoid(F)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
Xc, yc = make_moons(n_samples=600, noise=0.3, random_state=1)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.3, random_state=0)

gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=0)
gbc.fit(Xc_train, yc_train)
print("Train acc:", accuracy_score(yc_train, gbc.predict(Xc_train)))
print("Test acc :", accuracy_score(yc_test, gbc.predict(Xc_test)))

def log_loss(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

test_loss = [log_loss(yc_test, sigmoid(F)) for F in gbc.staged_decision_function(Xc_test)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(gbc, Xc_train, yc_train, "Gradient boosting classifier (200 trees)", ax=axes[0])
axes[1].plot(gbc.train_loss, label="train log-loss"); axes[1].plot(test_loss, label="test log-loss")
axes[1].set_xlabel("number of trees"); axes[1].set_ylabel("log-loss"); axes[1].legend()
axes[1].set_title(f"Loss curves (best test loss at {np.argmin(test_loss) + 1} trees)")
plt.tight_layout(); plt.show()

## Summary
* Gradient boosting fits each new tree to the **negative gradient** of the loss — any differentiable loss works.
* Key hyper-parameters: number of trees, learning rate, tree depth, subsample ratio. Use early stopping on a validation set.
* Bagging/RF reduce variance with deep independent trees; boosting reduces bias with shallow sequential trees.
* XGBoost, LightGBM and CatBoost are optimised implementations of this same idea (plus second-order terms and regularisation).